# Notebook 07: Notable Events

**One Sensor, One Year — Edition 1: India Breathes**

Every sensor tells a story through its extremes. What were the record days, the holidays, the anomalies? These become the annotations for The Anatomy.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df24 = pd.read_csv('../data/processed/india_2024_derived.csv', parse_dates=['date'])
df24['day_name'] = df24['date'].dt.strftime('%A')
df24['month'] = df24['date'].dt.month

fuel_cols = ['coal', 'lignite', 'gas', 'nuclear', 'hydro', 'wind', 'solar', 'other_re']

# Earth & Sky palette
palette = {
    'coal': '#3D2B1F', 'lignite': '#6B4226', 'gas': '#4A90D9',
    'nuclear': '#2EC4B6', 'hydro': '#1B4F72', 'wind': '#85C1E9',
    'solar': '#F5B041', 'other_re': '#A569BD',
}

## 1. Records Table — The Extremes of 2024

In [2]:
records = []

# Overall records
metrics = [
    ('total', 'Highest total generation'),
    ('total', 'Lowest total generation'),
    ('coal', 'Highest coal day'),
    ('coal', 'Lowest coal day'),
    ('hydro', 'Highest hydro day'),
    ('wind', 'Highest wind day'),
    ('solar', 'Highest solar day'),
    ('renewables', 'Highest renewables day'),
    ('clean_share', 'Cleanest grid day'),
    ('clean_share', 'Dirtiest grid day'),
    ('co2_total', 'Highest CO2 day'),
    ('co2_total', 'Lowest CO2 day'),
    ('emissions_intensity', 'Lowest emissions intensity'),
    ('emissions_intensity', 'Highest emissions intensity'),
]

print('RECORDS TABLE — India Grid 2024')
print('=' * 95)
print(f'{"Record":35s} {"Date":>12s} {"Day":>10s} {"Value":>12s}')
print('-' * 95)

for col, label in metrics:
    if 'Lowest' in label or 'Dirtiest' in label:
        idx = df24[col].idxmin()
    else:
        idx = df24[col].idxmax()
    row = df24.loc[idx]
    val = row[col]
    unit = '%' if 'share' in col or 'intensity' in col else (' kt' if 'co2' in col else ' MU')
    if 'intensity' in col:
        unit = ' tCO2/GWh'
    print(f'{label:35s} {row["date"].strftime("%b %d"):>12s} {row["day_name"]:>10s} {val:>10.1f}{unit}')
    records.append({
        'event': label, 'date': row['date'].strftime('%Y-%m-%d'),
        'day': row['day_name'], 'value': f'{val:.1f}{unit}'
    })

RECORDS TABLE — India Grid 2024
Record                                      Date        Day        Value
-----------------------------------------------------------------------------------------------
Highest total generation                  May 30   Thursday     5770.1 MU
Lowest total generation                   Nov 02   Saturday     4051.0 MU
Highest coal day                          May 04   Saturday     4054.0 MU
Lowest coal day                           Aug 25     Sunday     2814.8 MU
Highest hydro day                         Sep 03    Tuesday      746.9 MU
Highest wind day                          May 28    Tuesday      619.4 MU
Highest solar day                         Apr 30    Tuesday      425.6 MU
Highest renewables day                    May 28    Tuesday     1042.7 MU
Cleanest grid day                         Aug 26     Monday       36.8%
Dirtiest grid day                         Feb 03   Saturday       15.2%
Highest CO2 day                           May 04   Saturday    

## 2. Indian Holidays — Does the Grid Celebrate?

In [3]:
# Major Indian holidays in 2024
holidays = [
    ('2024-01-15', 'Makar Sankranti / Pongal'),
    ('2024-01-26', 'Republic Day'),
    ('2024-03-25', 'Holi'),
    ('2024-03-29', 'Good Friday'),
    ('2024-04-11', 'Eid ul-Fitr'),
    ('2024-04-14', 'Ambedkar Jayanti / Baisakhi'),
    ('2024-04-17', 'Ram Navami'),
    ('2024-04-21', 'Mahavir Jayanti'),
    ('2024-05-01', 'May Day'),
    ('2024-05-23', 'Buddha Purnima'),
    ('2024-06-17', 'Eid ul-Adha'),
    ('2024-07-17', 'Muharram'),
    ('2024-08-15', 'Independence Day'),
    ('2024-08-26', 'Janmashtami'),
    ('2024-09-16', 'Milad-un-Nabi'),
    ('2024-10-02', 'Gandhi Jayanti'),
    ('2024-10-12', 'Dussehra'),
    ('2024-10-31', 'Halloween / pre-Diwali'),
    ('2024-11-01', 'Diwali'),
    ('2024-11-02', 'Diwali day 2'),
    ('2024-11-15', 'Guru Nanak Jayanti'),
    ('2024-12-25', 'Christmas'),
]

print('Holiday Grid Impact — 2024')
print('=' * 105)
print(f'{"Holiday":30s} {"Date":>10s} {"Day":>10s} {"Total":>8s} {"vs Avg":>8s} {"Coal%":>7s} {"Clean%":>7s}')
print('-' * 105)

annual_avg = df24['total'].mean()

for date_str, name in holidays:
    row = df24[df24['date'] == date_str]
    if len(row) == 0:
        continue
    row = row.iloc[0]
    diff = (row['total'] / annual_avg - 1) * 100
    coal_pct = row['coal'] / row['total'] * 100
    clean_pct = row['clean_share']
    print(f'{name:30s} {date_str[5:]:>10s} {row["day_name"]:>10s} {row["total"]:8.0f} {diff:+7.1f}% {coal_pct:6.1f}% {clean_pct:6.1f}%')

Holiday Grid Impact — 2024
Holiday                              Date        Day    Total   vs Avg   Coal%  Clean%
---------------------------------------------------------------------------------------------------------
Makar Sankranti / Pongal            01-15     Monday     4402    -9.3%   79.8%   16.2%
Republic Day                        01-26     Friday     4531    -6.7%   78.1%   17.6%
Holi                                03-25     Monday     4274   -11.9%   76.2%   19.7%
Good Friday                         03-29     Friday     5041    +3.8%   77.8%   18.3%
Eid ul-Fitr                         04-11   Thursday     4927    +1.5%   77.9%   17.8%
Ambedkar Jayanti / Baisakhi         04-14     Sunday     4720    -2.8%   77.8%   18.2%
Ram Navami                          04-17  Wednesday     5135    +5.8%   76.4%   18.2%
Mahavir Jayanti                     04-21     Sunday     4969    +2.4%   76.5%   19.4%
May Day                             05-01  Wednesday     5138    +5.9%   73.4%   21.

## 3. The Diwali Effect

In [4]:
# Diwali week: Oct 28 - Nov 4
diwali_mask = (df24['date'] >= '2024-10-28') & (df24['date'] <= '2024-11-04')
diwali_week = df24[diwali_mask]

fig = go.Figure()

for col, label in zip(fuel_cols, ['Coal','Lignite','Gas','Nuclear','Hydro','Wind','Solar','Other RE']):
    fig.add_trace(go.Bar(
        x=diwali_week['date'].dt.strftime('%b %d (%a)'),
        y=diwali_week[col],
        name=label, marker_color=palette[col],
        hovertemplate=f'{label}: %{{y:.0f}} MU<extra></extra>',
    ))

fig.update_layout(
    title='Diwali Week Generation Mix (Oct 28 - Nov 4, 2024)',
    yaxis_title='Generation (MU/day)',
    barmode='stack',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0.5, xanchor='center'),
)
fig.show()

nov1 = df24[df24['date'] == '2024-11-01'].iloc[0]
nov2 = df24[df24['date'] == '2024-11-02'].iloc[0]
print(f'Diwali (Nov 1): {nov1["total"]:.0f} MU ({(nov1["total"]/annual_avg-1)*100:+.1f}% vs avg)')
print(f'Day after (Nov 2): {nov2["total"]:.0f} MU ({(nov2["total"]/annual_avg-1)*100:+.1f}% vs avg) — LOWEST day of 2024!')

Diwali (Nov 1): 4061 MU (-16.3% vs avg)
Day after (Nov 2): 4051 MU (-16.5% vs avg) — LOWEST day of 2024!


## 4. The Heatwave Peak — May/June

In [5]:
# Top 10 demand days
top10 = df24.nlargest(10, 'total')[['date', 'day_name', 'total', 'coal', 'renewables', 'clean_share', 'co2_total']]

print('Top 10 Highest Demand Days — 2024')
print('=' * 90)
print(f'{"#":>3s} {"Date":>12s} {"Day":>10s} {"Total":>8s} {"Coal":>8s} {"RE":>8s} {"Clean%":>7s} {"CO2 kt":>8s}')
print('-' * 90)
for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(f'{i:3d} {row["date"].strftime("%b %d"):>12s} {row["day_name"]:>10s} {row["total"]:8.0f} {row["coal"]:8.0f} {row["renewables"]:8.0f} {row["clean_share"]:6.1f}% {row["co2_total"]:8.0f}')

print(f'\nAll top 10 days are in: {", ".join(top10["date"].dt.strftime("%B").unique())}')

Top 10 Highest Demand Days — 2024
  #         Date        Day    Total     Coal       RE  Clean%   CO2 kt
------------------------------------------------------------------------------------------
  1       May 30   Thursday     5770     3941      883   26.3%     3935
  2       May 31     Friday     5704     3906      835   26.0%     3906
  3       May 29  Wednesday     5668     3692     1034   29.5%     3689
  4       Jun 18    Tuesday     5617     3864      819   25.7%     3856
  5       Jun 17     Monday     5595     4020      665   23.2%     3986
  6       May 24     Friday     5577     4003      670   23.0%     3989
  7       Jun 01   Saturday     5572     3862      793   25.3%     3855
  8       Jun 19  Wednesday     5537     3736      867   26.8%     3732
  9       Jun 15   Saturday     5529     3906      773   25.0%     3856
 10       May 28    Tuesday     5515     3638     1043   29.7%     3610

All top 10 days are in: May, June


In [6]:
# Bottom 10 demand days
bot10 = df24.nsmallest(10, 'total')[['date', 'day_name', 'total', 'coal', 'renewables', 'clean_share', 'co2_total']]

print('Bottom 10 Lowest Demand Days — 2024')
print('=' * 90)
print(f'{"#":>3s} {"Date":>12s} {"Day":>10s} {"Total":>8s} {"Coal":>8s} {"RE":>8s} {"Clean%":>7s} {"CO2 kt":>8s}')
print('-' * 90)
for i, (_, row) in enumerate(bot10.iterrows(), 1):
    print(f'{i:3d} {row["date"].strftime("%b %d"):>12s} {row["day_name"]:>10s} {row["total"]:8.0f} {row["coal"]:8.0f} {row["renewables"]:8.0f} {row["clean_share"]:6.1f}% {row["co2_total"]:8.0f}')

print(f'\nPattern: {", ".join(bot10["day_name"].unique())} — mostly holidays/weekends?')

Bottom 10 Lowest Demand Days — 2024
  #         Date        Day    Total     Coal       RE  Clean%   CO2 kt
------------------------------------------------------------------------------------------
  1       Nov 02   Saturday     4051     2956      457   23.3%     2926
  2       Nov 01     Friday     4061     2988      437   22.6%     2957
  3       Dec 01     Sunday     4062     3058      506   22.1%     2992
  4       Nov 03     Sunday     4068     2969      463   23.3%     2941
  5       Dec 29     Sunday     4197     3128      566   22.3%     3086
  6       Nov 17     Sunday     4214     3198      489   21.0%     3144
  7       Dec 02     Monday     4219     3185      530   22.0%     3111
  8       Mar 03     Sunday     4232     3146      559   21.6%     3124
  9       Nov 24     Sunday     4236     3254      451   20.4%     3185
 10       Dec 08     Sunday     4249     3238      492   20.9%     3180

Pattern: Saturday, Friday, Sunday, Monday — mostly holidays/weekends?


## 5. Anomaly Detection — Unusual Days

In [7]:
# Find days that deviate most from their monthly average
monthly_avg = df24.groupby('month')['total'].transform('mean')
df24['total_anomaly'] = (df24['total'] - monthly_avg) / monthly_avg * 100

# Biggest negative anomalies (unexpected drops)
drops = df24.nsmallest(10, 'total_anomaly')[['date', 'day_name', 'total', 'total_anomaly']]
print('Biggest Negative Anomalies (unexpected drops vs monthly avg):')
for _, row in drops.iterrows():
    print(f'  {row["date"].strftime("%b %d")} ({row["day_name"]:>9s}): {row["total"]:,.0f} MU ({row["total_anomaly"]:+.1f}%)')

print()

# Biggest positive anomalies (unexpected spikes)
spikes = df24.nlargest(10, 'total_anomaly')[['date', 'day_name', 'total', 'total_anomaly']]
print('Biggest Positive Anomalies (unexpected spikes vs monthly avg):')
for _, row in spikes.iterrows():
    print(f'  {row["date"].strftime("%b %d")} ({row["day_name"]:>9s}): {row["total"]:,.0f} MU ({row["total_anomaly"]:+.1f}%)')

Biggest Negative Anomalies (unexpected drops vs monthly avg):
  Mar 03 (   Sunday): 4,232 MU (-10.3%)
  Oct 31 ( Thursday): 4,318 MU (-10.2%)
  Mar 25 (   Monday): 4,274 MU (-9.4%)
  Jul 07 (   Sunday): 4,644 MU (-9.2%)
  Jun 30 (   Sunday): 4,875 MU (-9.0%)
  Dec 01 (   Sunday): 4,062 MU (-8.2%)
  Nov 02 ( Saturday): 4,051 MU (-7.8%)
  Nov 01 (   Friday): 4,061 MU (-7.6%)
  Oct 13 (   Sunday): 4,452 MU (-7.5%)
  Nov 03 (   Sunday): 4,068 MU (-7.4%)

Biggest Positive Anomalies (unexpected spikes vs monthly avg):
  Oct 04 (   Friday): 5,265 MU (+9.4%)
  May 30 ( Thursday): 5,770 MU (+8.3%)
  Sep 24 (  Tuesday): 5,380 MU (+8.3%)
  Sep 23 (   Monday): 5,364 MU (+8.0%)
  Oct 01 (  Tuesday): 5,188 MU (+7.8%)
  Oct 03 ( Thursday): 5,188 MU (+7.8%)
  Oct 08 (  Tuesday): 5,177 MU (+7.6%)
  Oct 09 (Wednesday): 5,174 MU (+7.6%)
  Oct 05 ( Saturday): 5,163 MU (+7.3%)
  May 31 (   Friday): 5,704 MU (+7.1%)


## 6. Event Timeline — Visual Summary

In [8]:
# Plot total generation with key events annotated
fig = go.Figure()

smooth = df24.set_index('date')['total'].rolling(7, center=True).mean()
fig.add_trace(go.Scatter(
    x=smooth.index, y=smooth,
    mode='lines', line=dict(width=2, color='#2C3E50'),
    name='Total (7d avg)',
    hovertemplate='%{y:.0f} MU | %{x|%b %d}<extra></extra>',
))

# Annotate key events
key_events = [
    ('2024-01-26', 'Republic Day', -40, -50),
    ('2024-03-25', 'Holi', 40, -40),
    ('2024-05-30', 'Peak demand\n(heatwave)', -50, -50),
    ('2024-06-01', 'Monsoon\nonset', 40, 40),
    ('2024-08-15', 'Independence\nDay', -40, -50),
    ('2024-08-26', 'Cleanest day\n(36.8% clean)', 40, 40),
    ('2024-09-30', 'Monsoon\nretreat', -40, 40),
    ('2024-11-01', 'Diwali', -40, -50),
    ('2024-11-02', 'Lowest day\n(post-Diwali)', 40, 40),
    ('2024-12-25', 'Christmas', 40, -40),
]

for date_str, label, ax, ay in key_events:
    ts = pd.Timestamp(date_str)
    if ts in smooth.index:
        y_val = smooth.loc[ts]
    else:
        y_val = df24[df24['date'] == date_str]['total'].iloc[0]
    fig.add_annotation(
        x=ts, y=y_val,
        text=label, showarrow=True, arrowhead=2,
        ax=ax, ay=ay, font=dict(size=9),
        bordercolor='#666', borderwidth=1, borderpad=3,
        bgcolor='rgba(255,255,255,0.8)',
    )

# Monsoon band
fig.add_shape(type='rect', x0=pd.Timestamp('2024-06-01'), x1=pd.Timestamp('2024-09-30'),
              y0=0, y1=1, yref='paper',
              fillcolor='rgba(100,149,237,0.08)', line_width=0)

fig.update_layout(
    title='India Grid 2024 — Annotated Timeline',
    yaxis_title='Total Generation (MU/day, 7d avg)',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=550,
    showlegend=False,
)
fig.show()

## 7. Events Export — For The Anatomy & Interactive

In [9]:
import json

# Compile all notable events for downstream use
events_list = [
    {'date': '2024-01-26', 'type': 'holiday', 'label': 'Republic Day'},
    {'date': '2024-03-25', 'type': 'holiday', 'label': 'Holi'},
    {'date': '2024-04-11', 'type': 'holiday', 'label': 'Eid ul-Fitr'},
    {'date': df24.loc[df24['total'].idxmax(), 'date'].strftime('%Y-%m-%d'),
     'type': 'record', 'label': f'Peak demand: {df24["total"].max():.0f} MU (heatwave)'},
    {'date': '2024-06-01', 'type': 'season', 'label': 'Monsoon onset'},
    {'date': '2024-08-15', 'type': 'holiday', 'label': 'Independence Day'},
    {'date': df24.loc[df24['clean_share'].idxmax(), 'date'].strftime('%Y-%m-%d'),
     'type': 'record', 'label': f'Cleanest grid: {df24["clean_share"].max():.1f}% clean'},
    {'date': '2024-09-30', 'type': 'season', 'label': 'Monsoon retreat'},
    {'date': '2024-10-12', 'type': 'holiday', 'label': 'Dussehra'},
    {'date': '2024-11-01', 'type': 'holiday', 'label': 'Diwali'},
    {'date': df24.loc[df24['total'].idxmin(), 'date'].strftime('%Y-%m-%d'),
     'type': 'record', 'label': f'Lowest demand: {df24["total"].min():.0f} MU (post-Diwali)'},
    {'date': '2024-12-25', 'type': 'holiday', 'label': 'Christmas'},
]

with open('../data/export/events_2024.json', 'w') as f:
    json.dump(events_list, f, indent=2)

print(f'Exported {len(events_list)} events to data/export/events_2024.json')
for e in events_list:
    print(f'  {e["date"]} [{e["type"]:>7s}] {e["label"]}')

Exported 12 events to data/export/events_2024.json
  2024-01-26 [holiday] Republic Day
  2024-03-25 [holiday] Holi
  2024-04-11 [holiday] Eid ul-Fitr
  2024-05-30 [ record] Peak demand: 5770 MU (heatwave)
  2024-06-01 [ season] Monsoon onset
  2024-08-15 [holiday] Independence Day
  2024-08-26 [ record] Cleanest grid: 36.8% clean
  2024-09-30 [ season] Monsoon retreat
  2024-10-12 [holiday] Dussehra
  2024-11-01 [holiday] Diwali
  2024-11-02 [ record] Lowest demand: 4051 MU (post-Diwali)
  2024-12-25 [holiday] Christmas


## Key Findings

1. **Peak demand: May 30** — heatwave pushed the grid to its absolute maximum
2. **Lowest demand: Nov 2** — the day after Diwali. India sleeps in.
3. **Diwali effect is real:** demand drops sharply on Diwali and especially the day after
4. **Heatwave days are also the dirtiest** — coal maxes out to meet cooling demand
5. **Aug 26 is the star day:** cleanest grid (36.8% clean) AND lowest emissions intensity
6. **Holiday drops aren't all equal:** Republic Day barely dips, but Diwali and Holi show clear drops
7. **Top 10 demand days cluster in May-June** — the heatwave is the grid's stress test
8. **Bottom 10 cluster around holidays and Sundays** — the grid rests when India rests

→ Next: Notebook 08 — Weekly Patterns